## BLIP-2 to automatically caption all of your photos.


In [1]:
import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

import os
from PIL import Image

In [2]:
def load_images(directory_path: str) -> list[str]:
    directory: list[str] = os.listdir(directory_path)
    list_of_photos = []
    for file in directory:
        list_of_photos.append(Image.open(directory_path + "/" + file))
    return list_of_photos

In [3]:
photos = load_images("Imagini")
photos

[<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=4608x3456>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x721>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=CMYK size=2592x1773>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1200x800>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=1280x720>]

In [4]:
torch.cuda.memory_allocated()

0

In [5]:
class TheCaptioner:
    def __init__(self, model_id) -> None:
        self.processor = Blip2Processor.from_pretrained(model_id)
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            model_id, device_map="auto", dtype=torch.bfloat16
        )
        self.default_prompt = "Question: What is shown in this image? Answer:"

    def caption_photo(self, image):
        # print(image)
        processed_inputs = self.processor(
            images=image, text=self.default_prompt, return_tensors="pt"
        ).to(device="cuda")
        # print(processed_inputs)
        generated_ids = self.model.generate(
            **processed_inputs,
            max_new_tokens=20,
        )
        print(generated_ids)
        generated_text = self.processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0].strip()
        return generated_text

In [6]:
torch.cuda.is_available()

True

In [7]:
captioner = TheCaptioner("Salesforce/blip2-opt-2.7b")

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

In [8]:
generated_caption = captioner.caption_photo(photos[0])

tensor([[50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265,
         50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265,
         50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265, 50265,
         50265, 50265,     2, 45641,    35,   653,    16,  2343,    11,    42,
          2274,   116, 31652,    35,    83,   313,  2828,    23,    10,  3034,
         50118]], device='cuda:0')


In [11]:
generated_caption[len(captioner.default_prompt ) + 1:]

'A man sitting at a computer'